# Qwen3.5-4B LoRA Fine-Tuning — Kaggle (T4 x2)

Fine-tune Qwen3.5-4B on one or more datasets published under
`michaelowusuntim6` on Hugging Face.

**Hardware:** Kaggle T4 x2, 32 GB VRAM total.
**Precision:** fp16. T4 does not support native bf16.
**Context:** 32768 tokens. Qwen3.5-4B was trained at 32K.
**Batch:** 1 per device, gradient accumulation 8.

Quota: 30 h/week GPU. An epoch on 10k records at 32K context takes roughly
60–90 minutes on T4 x2. Plan accordingly.

**Known constraint:** 32K context at 4B fp16 is at the edge of 32 GB. If you
OOM, drop MAX_SEQ_LEN to 16384 and re-run. Do not raise batch size above 1.


## Quota awareness

Check remaining accelerators with `!kaggle quota` (below) or the Kaggle UI.


In [ ]:
!kaggle quota 2>/dev/null || echo "kaggle CLI not available; check the UI"

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps transformers trl peft accelerate bitsandbytes datasets
!pip install -q --no-deps unsloth_zoo


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ── CONFIG ────────────────────────────────────────
DATASETS = [
    "code-review-qwen35",
    "debug-qwen35",
]

BASE_MODEL = "unsloth/Qwen3.5-4B"
EXPERT_NAME = "code-review-lora"
MAX_SEQ_LEN = 32768
LIMIT = 10000          # 0 = exhaust every mixed record
PUSH_TO_HUB = False
OUT_DIR = "/kaggle/working"
SEED = 3407

# Kaggle T4 x2 — fp16, 32K context, batch 1, grad accum 8.
print("GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("bf16 supported (will be False on T4):",
          torch.cuda.is_bf16_supported())


In [ ]:
from datasets import load_dataset
import sys, os

# Ensure the repo root is importable when running from Colab
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from orchestrator.dataset_mixer import smart_mix, smart_mix_report

# DATASETS is configured in the config cell:
#   str    -> single dataset, no mixing
#   list   -> exhaustively mix every record from every dataset
#
# The mixer sorts by size, uses whole-number ratios interleaved,
# then appends the truncated decimals in descending order so no
# record is lost. Deterministic given SEED.

if isinstance(DATASETS, str):
    ds = load_dataset(f"michaelowusuntim6/{DATASETS}", split="train")
    print(f"Single dataset: {len(ds)} records")
else:
    plan = smart_mix_report(DATASETS)
    print("Mix plan:")
    print(f"  sorted:   {plan['sorted']}")
    print(f"  smallest: {plan['smallest']}")
    print(f"  ratios:   {plan['ratios']}")
    print(f"  wholes:   {plan['wholes']}")
    print(f"  extras:   {plan['extras']}")
    print(f"  whole mix: {plan['whole_total']} records (interleaved)")
    print(f"  extras:    {plan['extras_total']} records (appended)")
    print(f"  final:     {plan['final_total']} records")
    ds = smart_mix(DATASETS, seed=SEED)
    print(f"Loaded {len(ds)} records")

if LIMIT and LIMIT < len(ds):
    ds = ds.select(range(LIMIT))
print(f"{len(ds)} training records after limit")

# Sanity: content must be a string (FastLanguageModel needs str)
sample = ds[0]["messages"][0]
print(f"First record content type: {type(sample['content']).__name__}")
assert isinstance(sample["content"], str), "content must be str, not list"


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,             # auto -> fp16 on T4
    load_in_4bit=False,     # bf16/fp16 LoRA, not QLoRA
    device_map="auto",      # uses both T4s
)

# CRITICAL: verify Unsloth's chunked loss is patched (issue #5441).
# Without this, the stock ForCausalLMLoss does logits.float() and
# allocates ~30 GB at 32K seq_len -> OOM.
assert "Unsloth" in str(model.loss_function), \
    f"loss not patched (issue #5441): {model.loss_function}\n" \
    "Upgrade unsloth and unsloth_zoo, then re-run."
print("loss patch verified:", type(model.loss_function).__name__)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


In [ ]:
def format_record(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(format_record, num_proc=2)
print(len(ds), "formatted records")


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,       # 1 — do not raise at 32K
        gradient_accumulation_steps=8,       # smooths gradients
        num_train_epochs=1,
        learning_rate=1e-4,                  # 1e-4 for 4B; 2e-4 too hot
        bf16=False,                          # T4: no bf16
        fp16=True,                           # T4: fp16
        max_grad_norm=0.3,                   # tighter than default 1.0
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        output_dir=f"{OUT_DIR}/outputs/{EXPERT_NAME}",
        save_strategy="epoch",
        save_total_limit=1,
        seed=SEED,
        report_to="none",
    ),
)
trainer.train()


In [ ]:
model.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
tokenizer.save_pretrained(f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print("Adapter saved to", f"{OUT_DIR}/adapters/{EXPERT_NAME}")
print(f"Adapter name: {EXPERT_NAME} (push to HF as michaelowusuntim6/{EXPERT_NAME})")
print("Kaggle wipes /kaggle/working after the session — push to HF "
      "or copy to a Kaggle Dataset to persist.")


In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import login
    import os
    login(token=os.environ["HF_TOKEN"])
    model.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}")
    tokenizer.push_to_hub(f"michaelowusuntim6/{EXPERT_NAME}")
    print(f"Pushed: https://huggingface.co/michaelowusuntim6/{EXPERT_NAME}")
else:
    print("PUSH_TO_HUB is False — set to True to upload")


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "Explain a NULL pointer dereference in the Linux kernel."}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
